# AVIGPT — Colab T4 serve + ngrok

1. Runtime → **GPU (T4)**  
2. Run cells in order  
3. Paste the printed ngrok URL into local `ui/index.html`  

Do **not** train in the same session as serve.

In [ ]:
# Cell 1 — clone + install
!git clone https://github.com/avir2105/avigpt.git
%cd avigpt
# If nested: %cd avigpt
!pip install -q -r requirements.txt
!nvidia-smi

In [ ]:
# Cell 2 — start API in background (keeps other cells runnable)
import os, subprocess, time, requests

os.environ["MODEL_NAME"] = "avi2135/avigpt2"
os.environ["MAX_NEW_TOKENS"] = "128"
os.environ["HOST"] = "0.0.0.0"
os.environ["PORT"] = "8000"

subprocess.Popen(["python", "-m", "serve.app"])

for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.ok:
            print("READY", r.json())
            break
    except Exception:
        pass
    time.sleep(2)
    if i % 10 == 0:
        print(f"waiting… {i*2}s")
else:
    raise RuntimeError("API not ready — check serve logs / VRAM")

In [ ]:
# Cell 3 — ngrok public URL
# Auth token: https://dashboard.ngrok.com/get-started/your-authtoken
# If token already saved on this runtime, skip set_auth_token.
!pip install -q pyngrok
from pyngrok import ngrok

# ngrok.set_auth_token("YOUR_TOKEN")  # uncomment once if needed
try:
    ngrok.kill()
except Exception:
    pass
url = ngrok.connect(8000, "http")
print(url)
print("Paste this into ui/index.html on your laptop.")

In [ ]:
# Cell 4 — smoke tests
import requests

base = "http://127.0.0.1:8000"
print("health", requests.get(f"{base}/health").json())
ask = requests.post(
    f"{base}/ask",
    json={"question": "Where was Avinash born?", "use_rag": True},
).json()
print("ask", ask.get("answer"))
print("sources", [s.get("source") for s in ask.get("sources") or []])
ver = requests.post(
    f"{base}/verify",
    json={"claim": "Avinashsingh Rajput was born in Maharashtra"},
).json()
print("verify", ver.get("verdict"), ver.get("trace"))

## Local UI

On your PC open `ui/index.html` → paste the ngrok URL → enable RAG → Send / Verify.

Keep this runtime alive while demoing. Record a short video (Week 4) so reviewers do not need your live GPU.